<!-- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building Synapsa, an AI-native
learning platform.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/flagships/humanoid-lab/lessons/F15-L08-sim-to-real-honesty/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/flagships/humanoid-lab/lessons/F15-L08-sim-to-real-honesty/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=flagships/humanoid-lab/lessons/F15-L08-sim-to-real-honesty/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy and matplotlib, which Colab, Kaggle,
Binder and Codespaces already have. The cell below installs `mujoco==3.13.0` and fetches the
files it needs beside it, and does nothing where they are already present. On Kaggle, switch
Internet on in the notebook's settings first; Kaggle allows that only for phone-verified
accounts.

In [ ]:
# --- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = [("mujoco", "mujoco==3.13.0")]            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = ["assets/SOURCE.md", "assets/balancer.xml"]    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/flagships/humanoid-lab/lessons/F15-L08-sim-to-real-honesty/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# F15-L08 · Sim-to-real, and what a free tier cannot teach you

**You will build:** a reality-gap bench — one rollout function that can inject control
latency and sensor noise, a parameter sweep that finds where each gap topples a working
controller, a domain-randomisation wrapper, and a hardware ledger that decides from
NVIDIA's published requirements what this course is unable to give you.

**Time:** ~60 minutes · **Runs on:** a laptop CPU, no GPU, no download
· **Prerequisites:** F15-L01 (mjModel/mjData, stepping), F15-L03 (state feedback and
gains), F15-L05 (rollouts as the unit of evaluation)

By the end you will be able to:
1. Implement a rollout that injects control latency and sensor noise, and measure how far
   each one degrades a balancing controller before it topples it.
2. Measure the fraction of a rollout spent against the torque limit, and show that MuJoCo
   clamps a command the controller is never told was clamped.
3. Randomise torso mass and joint dry friction and report survival over a held-out set.
4. Implement a domain-randomisation wrapper and measure whether it narrows the held-out gap.
5. Decide from NVIDIA's own requirements whether a named GPU can run Isaac Sim.

Every number in this notebook's output is computed by the code you run. The only figures
typed by a human are the hardware prices in section 10, and each one carries the URL it came
from and the date it was read.

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import hashlib
import math
import sys
import time
from pathlib import Path
from typing import Callable

import mujoco
import numpy as np

print("mujoco", mujoco.__version__, "· numpy", np.__version__)

MODEL_FILENAME = "balancer.xml"


def balancer_xml_path() -> Path:
    """Locate the balancer model that ships next to this notebook.

    There is no download branch. The model was written for this lesson and lives in
    `assets/`; if it is missing, the checkout is broken and saying so beats a silent fetch.
    """
    try:
        here = Path(__file__).resolve().parent
    except NameError:  # a notebook has no __file__
        here = Path.cwd()
    for candidate in (here / "assets" / MODEL_FILENAME,
                      here.parent / "assets" / MODEL_FILENAME,
                      Path.cwd() / "assets" / MODEL_FILENAME,
                      Path.cwd().parent / "assets" / MODEL_FILENAME):
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"{MODEL_FILENAME} not found next to this lesson. It ships in assets/ and is never "
        "downloaded; restore it from the lesson directory."
    )


def load_balancer():
    """Compile the balancer and hand back a fresh (model, data) pair."""
    model = mujoco.MjModel.from_xml_path(str(balancer_xml_path()))
    return model, mujoco.MjData(model)


MODEL, DATA = load_balancer()
TORSO_ID = mujoco.mj_name2id(MODEL, mujoco.mjtObj.mjOBJ_BODY, "torso")
BASE_TORSO_MASS = float(MODEL.body_mass[TORSO_ID])
CTRL_RANGE = MODEL.actuator_ctrlrange.copy()

# The task: reject the lean stored in the model's one keyframe, for this long, without
# exceeding this angle. All of it is read from the model or fixed here, never guessed.
HORIZON_STEPS = 500
FALL_ANGLE = 0.6
NOMINAL = {"mass_scale": 1.0, "frictionloss": 0.0, "delay_steps": 0, "noise_std": 0.0,
           "seed": 0}

print(f"model: nq={MODEL.nq} nv={MODEL.nv} nu={MODEL.nu} dt={MODEL.opt.timestep} s")
print(f"torso mass {BASE_TORSO_MASS:.1f} kg, total {MODEL.body_mass.sum():.1f} kg")
print(f"torque limits (N*m): ankle {CTRL_RANGE[0]}, hip {CTRL_RANGE[1]}")
print(f"horizon {HORIZON_STEPS} steps = {HORIZON_STEPS * MODEL.opt.timestep:.1f} s "
      f"of simulated time; a lean past {FALL_ANGLE} rad counts as fallen")


# True in a notebook and when this file is run as a script; False when the autograder
# imports it. Every check and demo below runs under this guard, so the cell you are sitting
# in reports on itself, while importing the lesson never runs anything.
_IS_MAIN = __name__ == "__main__"

# The five exercises, in the order the progress board in the last cell lists them.
_EXERCISES = {
    "exercise 1": "rollout",
    "exercise 2": "apply_parameters",
    "exercise 3": "evaluate",
    "exercise 4": "sample_conditions",
    "exercise 5": "can_run_isaac_sim",
}
# label -> "passed" | "failed" | "not started": the latest verdict of every check that has
# run. The progress board in the last cell reads it.
_STATUS: dict = {}


def _try(label: str, check: Callable[[], None], needs: tuple = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the others. Nothing is swallowed: every verdict is
    recorded in `_STATUS`, and the last cell exits non-zero if any check came back wrong when
    this file runs as a script.

    `needs` names the exercises a demo consumes. Until each of them has passed its own check,
    the demo says which one it is waiting for and skips, instead of failing on its behalf.
    """
    waiting = [n for n in needs if _STATUS.get(n) != "passed"]
    if waiting:
        _STATUS.pop(label, None)
        one = len(waiting) == 1
        names = [f"{n} ({_EXERCISES[n]})" for n in waiting] if len(waiting) <= 2 else waiting
        names = names[0] if one else ", ".join(names[:-1]) + " and " + names[-1]
        print(f"{label}: skipped — it needs {names} to pass first. Finish "
              f"{'that' if one else 'those'}, re-run "
              + ("its check cell" if one else "their check cells") + ", then re-run this one.")
        return
    try:
        check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        print(f"{label}: " + (f"not started yet — {exc}" if str(exc) else
                              "not implemented yet — fill in the stub above")
              + ", then re-run this cell.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. The gap is not one thing, it is four

"It worked in simulation" almost never fails for a single reason. Four separate gaps show
up on every real machine, and each has its own signature:

| gap | what the simulator assumes | what the robot does |
|---|---|---|
| **latency** | your torque applies this instant | it applies some milliseconds later |
| **saturation** | the motor delivers what you ask | it delivers what it has |
| **sensor noise** | you know the state exactly | you know an estimate of it |
| **parameters** | mass and friction are these numbers | they are nearly those numbers |

The rest of this lesson turns each one into a knob and measures it. Run the next cell to see
the controller working with every knob at zero — the sim-to-sim best case.

In [ ]:
# A state-feedback controller: torque = -K @ [ankle, hip, ankle_rate, hip_rate].
# These gains came out of the search in section 8; treat them as given for now.
BASELINE_GAINS = np.array([346.8, 52.6, 27.0, 6.7, -18.9, 54.9, -6.2, 4.1])


def reset_to_lean(model, data) -> None:
    """Put the machine back on the one keyframe the model ships: leaning and falling."""
    mujoco.mj_resetDataKeyframe(model, data, 0)
    mujoco.mj_forward(model, data)


reset_to_lean(MODEL, DATA)
print(f"start pose: ankle {DATA.qpos[0]:+.3f} rad, hip {DATA.qpos[1]:+.3f} rad, "
      f"ankle rate {DATA.qvel[0]:+.3f} rad/s")
print("it is already falling forwards; the controller has to catch it")

## 2. Exercise 1 — `rollout`, the instrument everything else is built on

This is the one function that matters. It runs the controller for the full horizon and
reports four things: did it survive, for how many steps, how badly it wobbled, and what
fraction of the time it was asking for more torque than the motor has.

Two of the four gaps live inside this loop. **Latency** is a pipeline: the torque you apply
now was computed several steps ago. **Sensor noise** corrupts what the controller *sees*,
never the true state — that distinction is the whole point, so read it twice.

<details><summary>💡 Hint 1 — what to think about</summary>

Keep apart three things the loop exists to separate: the TRUE state, which only MuJoCo
changes; what the controller SEES, the true state plus noise; and what the motor RECEIVES,
a command from some steps ago that may be beyond its limit. Most wrong answers blur two of
them. Then ask: with a delay of three steps, which commands are the first three applied?

</details>

<details><summary>💡 Hint 2 — the approach, in words</summary>

Before the loop: reset to the lean, build the generator once, and fill the pipeline with as
many zero commands as the delay. Each step: observe (the noise goes on a copy, never on
`data`), compute the command with its minus sign, append it and take the oldest off the
front, test that RAW command against the range, write it unclamped, step. Only then add the
squared ankle angle and test the absolute lean. On a fall, count the step you just took —
that count is also what the saturation fraction divides by.

</details>

In [ ]:
def rollout(model, data, gains, delay_steps: int = 0, noise_std: float = 0.0,
            seed: int = 0) -> dict:
    """Run the balancing controller for one episode and report how it went.

    The loop, in order, for each of `HORIZON_STEPS` steps:

    1. Read the TRUE state, `np.concatenate([data.qpos, data.qvel])` — four numbers.
    2. If `noise_std` is positive, form the OBSERVED state by adding
       `rng.normal(0.0, noise_std, size=4)` to it. The true state is never modified; only
       the controller's view of it is. Use `rng = np.random.default_rng(seed)`, created once
       before the loop, so a given `seed` replays exactly.
    3. Compute the command `u = -K @ observed`, where `K` is `gains` reshaped to (2, 4).
    4. Push `u` onto the end of a `pipeline` list that STARTED as `delay_steps` zero
       commands, then apply `pipeline.pop(0)`. With `delay_steps=0` you pop the command you
       just pushed, so nothing is delayed; with `delay_steps=3` the command you apply was
       computed three steps ago.
    5. Before writing it, check whether the applied command lies outside
       `model.actuator_ctrlrange` on either actuator; if so, count this step as saturated.
       Write the command UNCLAMPED to `data.ctrl` — MuJoCo does the clamping, and watching
       it do so is the point of section 3.
    6. `mujoco.mj_step(model, data)`, then accumulate `data.qpos[0] ** 2` for the RMS, and
       stop early if `abs(data.qpos[0]) > FALL_ANGLE`.

    Call `reset_to_lean(model, data)` first so every episode starts from the same pose.

    Example (the baseline gains, every gap switched off):
        >>> model, data = load_balancer()
        >>> r = rollout(model, data, BASELINE_GAINS)
        >>> r["survived"], r["steps"]
        (True, 500)

    Returns:
        dict with exactly these four keys:
          "survived"            bool, True only if it lasted the whole horizon
          "steps"               int, steps actually taken
          "rms_lean"            float, sqrt(mean(ankle angle squared)) over the steps taken,
                                or float("inf") if it fell — a fallen robot has no tracking
                                error, it has no robot
          "saturated_fraction"  float in [0, 1], saturated steps / steps taken
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_rollout() -> None:
    model, data = load_balancer()
    r = rollout(model, data, BASELINE_GAINS)
    assert set(r) == {"survived", "steps", "rms_lean", "saturated_fraction"}, (
        f"keys were {sorted(r)} — return a dict with exactly those four names."
    )
    assert r["survived"] and r["steps"] == HORIZON_STEPS, (
        f"the baseline gains survived={r['survived']} for {r['steps']} steps with every gap "
        "switched off. They are known good, so the loop is wrong: the usual cause is "
        "forgetting reset_to_lean, or applying +K @ x instead of -K @ x."
    )
    assert 0.0 < r["rms_lean"] < 0.2, (
        f"rms_lean was {r['rms_lean']} — it is the root MEAN square of data.qpos[0] over the "
        "steps taken, so divide by the step count before taking the square root."
    )
    assert 0.0 < r["saturated_fraction"] < 1.0, (
        f"saturated_fraction was {r['saturated_fraction']} — at these gains the ankle spends "
        "part, but not all, of the episode against its 40 N*m limit. Exactly 0.0 means you "
        "compared the CLAMPED command against the range instead of the raw one."
    )
    took = data.time / model.opt.timestep
    assert abs(took - HORIZON_STEPS) < 1e-6, (
        f"you reported {r['steps']} steps, but MuJoCo's own clock says {took:.0f} steps were "
        f"simulated. The loop runs range({HORIZON_STEPS}) — check the bound — and `steps` is "
        "the number of steps actually taken, not a label."
    )
    slow = rollout(model, data, BASELINE_GAINS, delay_steps=2)
    assert slow["rms_lean"] > r["rms_lean"] * 1.3, (
        f"8 ms of latency changed rms_lean from {r['rms_lean']:.4f} to {slow['rms_lean']:.4f} "
        "— that is too little. Your pipeline is not actually delaying: it must START with "
        "delay_steps zeros, so the first commands applied are stale."
    )
    quiet = rollout(model, data, BASELINE_GAINS, noise_std=0.05, seed=1)
    again = rollout(model, data, BASELINE_GAINS, noise_std=0.05, seed=1)
    assert quiet["rms_lean"] == again["rms_lean"], (
        "the same seed gave two different answers — build the Generator once, before the "
        "loop, not inside it."
    )
    print(f"exercise 1 looks right: clean rms {r['rms_lean']:.4f}, "
          f"saturated {r['saturated_fraction']:.0%} of the episode")


if _IS_MAIN:
    _try("exercise 1", _check_rollout)

## 3. Saturation: the lie the controller is never told

MuJoCo clamps a command outside `ctrlrange` and says nothing. `data.ctrl` still holds the
absurd number you wrote; `data.actuator_force` holds what the motor actually produced. A
controller that reads back its own `ctrl` believes it got what it asked for.

Run this. Nothing here is typed — the clamp is measured.

In [ ]:
_m, _d = load_balancer()
_d.ctrl[:] = [1e6, 1e6]
mujoco.mj_step(_m, _d)
print(f"wrote ctrl      = {_d.ctrl}")
print(f"motor delivered = {_d.actuator_force}   <- clamped to ctrlrange, silently")
print(f"the ankle's real ceiling is {CTRL_RANGE[0][1]:.0f} N*m, and the controller "
      "cannot tell from data.ctrl that it ever hit it")

## 4. Exercise 2 — `apply_parameters`, and the hazard of a mutable model

The other two gaps are *model* edits, not loop edits. `mjModel` is mutable, which is what
makes randomisation cheap — you do not recompile the XML a thousand times, you edit the
compiled model in place.

That is also the trap. Edit it in place and forget to restore it, and every later condition
silently inherits the previous one's mass. Always set from the stored baseline, never
multiply what is already there.

<details><summary>💡 Hint 1 — what to think about</summary>

Call it twice with the same arguments and the model must end up the same — that is the
whole contract, and anything that reads the CURRENT mass to work out the new one breaks it.
Notice too that a call which mentions only friction still sets the mass, back to its default.

</details>

<details><summary>💡 Hint 2 — the approach, in words</summary>

Work the torso mass out from the constant stored at start-up, never from the model; write
the friction into every joint's entry, not just one; then ask MuJoCo to rebuild what it
derived from the old mass. Three statements, and none reads a value a previous call left.

</details>

In [ ]:
def apply_parameters(model, data, mass_scale: float = 1.0,
                     frictionloss: float = 0.0) -> None:
    """Set the torso mass and the joint dry friction on a compiled model, in place.

    - `model.body_mass[TORSO_ID]` becomes `BASE_TORSO_MASS * mass_scale`. Set it FROM the
      stored baseline; `model.body_mass[TORSO_ID] *= mass_scale` compounds across calls and
      is the single most common bug in this lesson.
    - `model.dof_frictionloss[:]` becomes `frictionloss`. This is MuJoCo's dry friction: an
      upper limit on the force friction can generate, applied to both joints.
    - Finally call `mujoco.mj_setConst(model, data)` so the quantities MuJoCo DERIVED from
      the old mass are rebuilt: `body_subtreemass`, `dof_M0`, `body_invweight0` and friends.
      Section 11 has a cell that shows one of them going stale. Be precise about what this
      costs you on this model: MuJoCo rebuilds the mass matrix every step, so a rollout here
      is unchanged either way — it is the derived bookkeeping that rots, which is why the
      grader checks `body_subtreemass` rather than a rollout.

    Example:
        >>> model, data = load_balancer()
        >>> apply_parameters(model, data, mass_scale=1.5)
        >>> round(float(model.body_mass[TORSO_ID]), 1)
        21.0
        >>> apply_parameters(model, data, mass_scale=1.0)   # restores, does not compound
        >>> round(float(model.body_mass[TORSO_ID]), 1)
        14.0

    Returns:
        None. This function mutates `model`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_apply_parameters() -> None:
    model, data = load_balancer()
    apply_parameters(model, data, mass_scale=1.5)
    got = float(model.body_mass[TORSO_ID])
    assert abs(got - BASE_TORSO_MASS * 1.5) < 1e-9, (
        f"torso mass is {got:.3f}, expected {BASE_TORSO_MASS * 1.5:.3f}."
    )
    apply_parameters(model, data, mass_scale=1.5)
    twice = float(model.body_mass[TORSO_ID])
    assert abs(twice - got) < 1e-9, (
        f"calling it twice with the same scale gave {got:.3f} then {twice:.3f} — you are "
        "multiplying the current mass instead of setting it from BASE_TORSO_MASS."
    )
    apply_parameters(model, data, frictionloss=3.0)
    assert np.allclose(model.dof_frictionloss, 3.0), (
        f"dof_frictionloss is {model.dof_frictionloss} — set every entry, and note this call "
        "also restores mass_scale to its default of 1.0."
    )
    assert abs(float(model.body_mass[TORSO_ID]) - BASE_TORSO_MASS) < 1e-9, (
        "that last call left mass_scale at its default of 1.0, so the torso mass should be "
        "back at the baseline; it is not, so you are not setting from the baseline."
    )
    print("exercise 2 looks right: mass and friction set from baseline, no compounding")


if _IS_MAIN:
    _try("exercise 2", _check_apply_parameters)

## 5. Exercise 3 — `evaluate`, one condition end to end

A *condition* is one dict describing one imagined robot: `mass_scale`, `frictionloss`,
`delay_steps`, `noise_std`, `seed`. Two of those keys are model edits and three are loop
arguments. Splitting them correctly is the exercise.

<details><summary>💡 Hint 1 — what to think about</summary>

Sort the five keys into two piles: those that change the machine, and those that change how
the episode is run. Then picture the model as the NEXT condition finds it, if this one had a
heavy torso — the check runs exactly that sequence on one model.

</details>

<details><summary>💡 Hint 2 — the approach, in words</summary>

Call `apply_parameters` every time — nominal included — with this condition's mass and
friction, so nothing from the previous condition survives. Then run the rollout with this
condition's delay, noise and seed as keyword arguments, and return its dict untouched.

</details>

In [ ]:
def evaluate(model, data, gains, condition: dict) -> dict:
    """Apply one condition to the model and run one rollout under it.

    `condition` has exactly the five keys of `NOMINAL`. Send `mass_scale` and `frictionloss`
    to `apply_parameters`; send `delay_steps`, `noise_std` and `seed` to `rollout`. Return
    the rollout's dict unchanged.

    Example:
        >>> model, data = load_balancer()
        >>> evaluate(model, data, BASELINE_GAINS, NOMINAL)["survived"]
        True
        >>> heavy = dict(NOMINAL, mass_scale=1.3)
        >>> evaluate(model, data, BASELINE_GAINS, heavy)["survived"]
        False

    Returns:
        The dict from `rollout`, with the same four keys.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def survival_rate(model, data, gains, conditions: list) -> float:
    """Fraction of `conditions` the controller survives. Provided; uses your `evaluate`."""
    if not conditions:
        return 0.0
    return sum(bool(evaluate(model, data, gains, c)["survived"])
               for c in conditions) / len(conditions)


def _check_evaluate() -> None:
    model, data = load_balancer()
    assert evaluate(model, data, BASELINE_GAINS, NOMINAL)["survived"], (
        "the nominal condition is the one the baseline gains were tuned on; it must survive. "
        "Check you are passing noise_std and delay_steps through rather than dropping them."
    )
    heavy = evaluate(model, data, BASELINE_GAINS, dict(NOMINAL, mass_scale=1.3))
    assert not heavy["survived"], (
        "a 30% heavier torso must topple these gains — if it survived, mass_scale is not "
        "reaching apply_parameters."
    )
    back = evaluate(model, data, BASELINE_GAINS, NOMINAL)
    assert back["survived"], (
        "the nominal condition failed straight after a heavy one: the model kept the heavy "
        "mass. apply_parameters must set from the baseline on every call."
    )
    print("exercise 3 looks right: conditions apply, and they do not leak into each other")


# evaluate hands its work to your rollout and apply_parameters, so it waits for both.
if _IS_MAIN:
    _try("exercise 3", _check_evaluate, needs=("exercise 1", "exercise 2"))

## 6. Where each gap actually bites

Now the measurement. Each sweep below turns exactly one knob with the baseline gains, and
prints what your own `rollout` reports. Read the saturation column as carefully as the
survival column — it is usually the first thing to move.

In [ ]:
def sweep(name: str, key: str, values, model=None, data=None) -> list:
    """Turn one knob across `values`, holding the rest at NOMINAL, and print the result."""
    if model is None:
        model, data = load_balancer()
    rows = []
    print(f"\n{name}")
    print(f"  {'value':>10s}  {'survived':>8s} {'steps':>6s} {'rms lean':>9s} {'saturated':>9s}")
    for v in values:
        r = evaluate(model, data, BASELINE_GAINS, dict(NOMINAL, **{key: v}))
        rows.append((v, r))
        rms = "fell" if r["rms_lean"] == float("inf") else f"{r['rms_lean']:.4f}"
        print(f"  {v:>10} {str(r['survived']):>9s} {r['steps']:>6d} {rms:>9s} "
              f"{r['saturated_fraction']:>8.0%}")
    return rows


def _check_sweeps() -> None:
    dt_ms = MODEL.opt.timestep * 1000
    lat = sweep(f"LATENCY (one step = {dt_ms:.0f} ms)", "delay_steps", [0, 2, 4, 8, 12, 16, 20])
    mass = sweep("TORSO MASS (multiplier)", "mass_scale", [0.8, 1.0, 1.1, 1.2, 1.3])
    fric = sweep("DRY FRICTION (N*m)", "frictionloss", [0.0, 5.0, 10.0, 20.0, 30.0])
    noise = sweep("SENSOR NOISE (rad, std)", "noise_std", [0.0, 0.02, 0.08, 0.12, 0.20])

    def first_fall(rows):
        for v, r in rows:
            if not r["survived"]:
                return v
        return None

    print(f"\nfirst latency that topples it : {first_fall(lat)} steps "
          f"({(first_fall(lat) or 0) * dt_ms:.0f} ms)")
    print(f"first mass multiplier that does: {first_fall(mass)}")
    print(f"first friction that does       : {first_fall(fric)}  "
          "<- dry friction degrades tracking without toppling it")
    print(f"first noise level that does    : {first_fall(noise)}")
    assert first_fall(lat) is not None, "latency should eventually topple it"
    assert first_fall(mass) is not None, "a heavy enough torso should topple it"
    assert first_fall(noise) is not None, "enough sensor noise should topple it"
    # The sentence printed below is a claim about this machine, so it is asserted rather
    # than merely typed: if dry friction ever did topple it, the prose would be a lie.
    assert first_fall(fric) is None, (
        f"dry friction toppled the machine at {first_fall(fric)} N*m, so the conclusion "
        "printed below no longer holds — re-read the friction table before trusting it"
    )
    assert fric[0][1]["rms_lean"] < fric[-1][1]["rms_lean"], (
        "friction should make the wobble worse even where it does not cause a fall"
    )
    # The multiple is COMPUTED from the table you just ran, never typed into the prose. If
    # the model or the gains ever change, this sentence changes with them.
    wobble_ratio = fric[-1][1]["rms_lean"] / fric[0][1]["rms_lean"]
    print(f"\nread that again: friction never topples it, and still multiplies the wobble by "
          f"{wobble_ratio:.1f}x while pinning the ankle at its limit for "
          f"{fric[-1][1]['saturated_fraction']:.0%} of the episode."
          "\nSurvival is not the only thing worth measuring, and it is often the last to move.")


if _IS_MAIN:
    _try("the sweeps", _check_sweeps,
         needs=("exercise 1", "exercise 2", "exercise 3"))

## 7. Exercise 4 — the domain-randomisation wrapper

Domain randomisation is one idea: instead of tuning against the simulator you have, tune
against a *distribution* of simulators, so the real one is just another draw. It is due to
Tobin et al. (2017) for appearance and Peng et al. (2017) for dynamics — this lesson
randomises dynamics, so it is the Peng variant. Both are sourced in `claims.yaml`.

Your job is the sampler. Draw the numbers in exactly the documented order, so that a given
seed reproduces exactly, for you and for the grader.

<details><summary>💡 Hint 1 — what to think about</summary>

The same seed must give the same list, for you and for the grader, which draws its own copy
in the documented order — so the ORDER of the five draws matters as much as their ranges.
And ask where each condition's own rollout seed comes from: not the `seed` you were handed.

</details>

<details><summary>💡 Hint 2 — the approach, in words</summary>

One generator, made once from the seed, before the loop. For each condition, five draws in
the order the docstring lists, each turned into a plain Python float or int as it comes out
— a numpy integer is not an int. Read the ranges from the `spec` argument, not from the
module-level table. The fifth draw is that condition's rollout seed.

</details>

In [ ]:
DR_SPEC = {
    "mass_scale": (0.85, 1.30),
    "frictionloss": (0.0, 15.0),
    "delay_steps": (0, 9),        # integers, upper bound exclusive
    "noise_std": (0.0, 0.03),
}
HELD_OUT_SEED, HELD_OUT_N = 4242, 40
DR_TRAIN_SEED, DR_TRAIN_N = 707, 10
TUNE_SEED = 1


def sample_conditions(n: int, seed: int, spec: dict = DR_SPEC) -> list:
    """Draw `n` random conditions from `spec`. This is the domain-randomisation wrapper.

    Create ONE generator, `rng = np.random.default_rng(seed)`, then build each condition by
    drawing in exactly this order — the order is part of the contract, because it is what
    makes a seed reproducible:

        mass_scale   = float(rng.uniform(*spec["mass_scale"]))
        frictionloss = float(rng.uniform(*spec["frictionloss"]))
        delay_steps  = int(rng.integers(*spec["delay_steps"]))
        noise_std    = float(rng.uniform(*spec["noise_std"]))
        seed         = int(rng.integers(0, 10000))

    Each condition gets its OWN rollout seed from the same generator, so two conditions with
    the same noise level still see different noise.

    Example:
        >>> c = sample_conditions(3, seed=0)
        >>> len(c), sorted(c[0])
        (3, ['delay_steps', 'frictionloss', 'mass_scale', 'noise_std', 'seed'])
        >>> sample_conditions(3, seed=0) == sample_conditions(3, seed=0)
        True

    Returns:
        A list of `n` dicts, each with the same five keys as `NOMINAL`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def ensemble_cost(model, data, gains, conditions: list) -> float:
    """Mean cost over conditions: wobble if it survived, a graded penalty if it fell.

    Provided. The penalty stays graded — falling at step 400 scores better than falling at
    step 10 — so the search always has a gradient to follow, even before anything survives.
    """
    total = 0.0
    for c in conditions:
        r = evaluate(model, data, gains, c)
        total += r["rms_lean"] if r["survived"] else 10.0 - 8.0 * (r["steps"] / HORIZON_STEPS)
    return total / len(conditions)


def cem_tune(conditions: list, seed: int = TUNE_SEED, iterations: int = 8,
             population: int = 40, elite: int = 8) -> np.ndarray:
    """Cross-entropy search for gains that do well across `conditions`. Provided.

    Nothing clever: sample gains from a Gaussian, keep the best few, refit, repeat. The only
    thing that changes between a nominally-tuned and a domain-randomised controller is the
    LIST OF CONDITIONS you hand it. That is the entire technique.
    """
    model, data = load_balancer()
    rng = np.random.default_rng(seed)
    mu = np.array([150.0, 30.0, 45.0, 8.0, -10.0, 50.0, -4.0, 10.0])
    sd = np.abs(mu) * 0.7 + 4.0
    for _ in range(iterations):
        candidates = rng.normal(mu, sd, size=(population, 8))
        scores = np.array([ensemble_cost(model, data, g, conditions) for g in candidates])
        keep = candidates[np.argsort(scores)[:elite]]
        mu, sd = keep.mean(axis=0), keep.std(axis=0) + 1e-3
    return mu


def _check_sample_conditions() -> None:
    got = sample_conditions(5, seed=0)
    assert len(got) == 5, f"asked for 5 conditions, got {len(got)}"
    assert sorted(got[0]) == sorted(NOMINAL), (
        f"a condition has keys {sorted(got[0])}, expected {sorted(NOMINAL)}"
    )
    assert got == sample_conditions(5, seed=0), (
        "the same seed produced different conditions — build the Generator once, from the "
        "seed you were given, and draw in the documented order."
    )
    assert got != sample_conditions(5, seed=1), "different seeds must give different draws"
    for c in got:
        assert DR_SPEC["mass_scale"][0] <= c["mass_scale"] <= DR_SPEC["mass_scale"][1], (
            f"mass_scale {c['mass_scale']} is outside {DR_SPEC['mass_scale']}"
        )
        assert isinstance(c["delay_steps"], int), (
            f"delay_steps is {type(c['delay_steps']).__name__}; it indexes a list, so make "
            "it an int with int(), not a numpy integer from rng.integers"
        )
    assert len({c["seed"] for c in got}) > 1, (
        "every condition got the same rollout seed — draw a fresh one per condition from the "
        "same generator, do not reuse the argument `seed`"
    )
    print("exercise 4 looks right: reproducible, in range, one rollout seed per condition")


if _IS_MAIN:
    _try("exercise 4", _check_sample_conditions)

## 8. Does it actually narrow the gap?

The honest test of domain randomisation is a parameter set the controller was never tuned
on. Below, two controllers are tuned — one against the nominal model alone, one against a
randomised ensemble — and both are scored on the same held-out conditions, drawn from a
seed neither search ever saw.

In [ ]:
def compare_nominal_vs_dr() -> dict:
    """Tune two controllers, score both on held-out conditions, return the comparison."""
    model, data = load_balancer()
    held_out = sample_conditions(HELD_OUT_N, seed=HELD_OUT_SEED)
    train_dr = sample_conditions(DR_TRAIN_N, seed=DR_TRAIN_SEED)

    t0 = time.perf_counter()
    gains_nominal = cem_tune([NOMINAL])
    t1 = time.perf_counter()
    gains_dr = cem_tune(train_dr)
    t2 = time.perf_counter()

    out = {
        "nominal_on_nominal": survival_rate(model, data, gains_nominal, [NOMINAL]),
        "nominal_on_held_out": survival_rate(model, data, gains_nominal, held_out),
        "dr_on_held_out": survival_rate(model, data, gains_dr, held_out),
        "tune_seconds_nominal": t1 - t0,
        "tune_seconds_dr": t2 - t1,
    }
    out["gap_closed"] = out["dr_on_held_out"] - out["nominal_on_held_out"]
    return out


def _check_comparison() -> None:
    r = compare_nominal_vs_dr()
    print(f"tuned on the nominal model alone   ({r['tune_seconds_nominal']:.1f}s)")
    print(f"tuned on {DR_TRAIN_N} randomised models     ({r['tune_seconds_dr']:.1f}s)")
    print(f"\n  nominal-tuned, scored on the model it was tuned on : "
          f"{r['nominal_on_nominal']:.0%}")
    print(f"  nominal-tuned, scored on {HELD_OUT_N} held-out models      : "
          f"{r['nominal_on_held_out']:.0%}")
    print(f"  DR-tuned,      scored on the same held-out models  : {r['dr_on_held_out']:.0%}")
    print(f"\n  gap closed by domain randomisation: {r['gap_closed']:+.1%}")
    assert r["nominal_on_nominal"] == 1.0, (
        "the nominally-tuned controller should be perfect on the model it was tuned on — "
        "that is exactly what makes its held-out score so instructive"
    )
    assert r["dr_on_held_out"] > r["nominal_on_held_out"], (
        f"DR scored {r['dr_on_held_out']:.0%} against nominal's {r['nominal_on_held_out']:.0%} "
        "on held-out conditions. Expected DR to win — check that cem_tune is receiving the "
        "randomised list and not [NOMINAL]."
    )
    print("\n  100% on the model you tuned on is not a result. It is the definition of the"
          "\n  model you tuned on. Only the held-out column is evidence of anything.")


if _IS_MAIN:
    _try("the held-out comparison", _check_comparison,
         needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4"))

## 9. Exercise 5 — what this course cannot give you

Everything above ran on a CPU because the whole flagship is built to. The moment you want
photoreal rendering, RTX sensor simulation or massively parallel GPU training — Isaac Sim
and Isaac Lab — the hardware requirement is real and published, and no amount of course
design gets around it.

Here is the part most write-ups get wrong, so this lesson makes you compute it rather than
repeat it. The usual claim is that free notebook GPUs "lack ray-tracing cores". **The T4
has RT Cores** (NVIDIA's own product page says so) and its 16 GB meets the 16 GB floor
exactly. The L4 clears both bars more comfortably still, with third-generation RT Cores and
24 GB. The real obstacles are different, and your function has to name them precisely.

<details><summary>💡 Hint 1 — what to think about</summary>

Three independent checks, each able to add one reason: do not stop at the first failure.
Watch the boundary — a card with exactly the minimum VRAM meets the minimum. And the grader
hands you a relaxed requirement, so every threshold must come from the argument.

</details>

<details><summary>💡 Hint 2 — the approach, in words</summary>

Start an empty list. Test for RT Cores only when the requirement asks for them; test VRAM
against the requirement's minimum with a strict less-than, quoting both numbers; test
whether NVIDIA names the card. Append in that order. `ok` is whether the list stayed empty.

</details>

In [ ]:
# Every field below is from a primary source, recorded in claims.yaml with its access date.
ISAAC_MIN = {
    "min_vram_gib": 16,
    "requires_rt_cores": True,
    "reference_gpu": "GeForce RTX 4080",
    "source": "https://docs.isaacsim.omniverse.nvidia.com/5.1.0/installation/requirements.html",
}

GPU_LEDGER = [
    # name, VRAM GiB, has RT cores, named in NVIDIA's requirements table, where you meet it
    {"name": "GeForce RTX 4080", "vram_gib": 16, "rt_cores": True, "named_by_nvidia": True,
     "found_on": "a workstation you bought"},
    {"name": "RTX PRO 6000 Blackwell", "vram_gib": 48, "rt_cores": True,
     "named_by_nvidia": True, "found_on": "a workstation you really bought"},
    {"name": "NVIDIA T4", "vram_gib": 16, "rt_cores": True, "named_by_nvidia": False,
     "found_on": "Colab free tier, when you are given one at all"},
    {"name": "NVIDIA L4", "vram_gib": 24, "rt_cores": True, "named_by_nvidia": False,
     "found_on": "cloud notebook tiers"},
    {"name": "NVIDIA A100", "vram_gib": 40, "rt_cores": False, "named_by_nvidia": False,
     "found_on": "research clusters; named by NVIDIA as unsupported"},
    {"name": "this machine's CPU", "vram_gib": 0, "rt_cores": False, "named_by_nvidia": False,
     "found_on": "where this entire flagship runs"},
]


def can_run_isaac_sim(gpu: dict, requirement: dict = ISAAC_MIN) -> dict:
    """Decide whether one GPU meets NVIDIA's published Isaac Sim requirements.

    Check all three, appending a reason for each failure IN THIS ORDER:

    1. `requirement["requires_rt_cores"]` is true and `gpu["rt_cores"]` is false ->
       a reason mentioning RT Cores
    2. `gpu["vram_gib"] < requirement["min_vram_gib"]` ->
       a reason quoting both numbers
    3. `gpu["named_by_nvidia"]` is false ->
       a reason saying it is not named in NVIDIA's requirements table

    Read `requirement`, never the `ISAAC_MIN` global — the grader relaxes the requirement to
    check you did.

    Note what this ordering forces you to notice: the T4 passes checks 1 and 2 and fails only
    check 3. The popular explanation for why free tiers cannot run Isaac Sim is the wrong one.

    Example:
        >>> can_run_isaac_sim(GPU_LEDGER[0])["ok"]
        True
        >>> len(can_run_isaac_sim(GPU_LEDGER[2])["reasons"])   # the T4 fails one check
        1
        >>> len(can_run_isaac_sim(GPU_LEDGER[4])["reasons"])   # the A100 fails two
        2

    Returns:
        dict with "ok" (bool, true only when there are no reasons) and "reasons" (list of str).
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_can_run_isaac_sim() -> None:
    print(f"{'GPU':<26s} {'VRAM':>5s} {'RT':>6s} {'runs Isaac Sim?':>16s}   why not")
    for gpu in GPU_LEDGER:
        v = can_run_isaac_sim(gpu)
        why = "" if v["ok"] else "; ".join(v["reasons"])
        print(f"{gpu['name']:<26s} {gpu['vram_gib']:>4d}G {str(gpu['rt_cores']):>6s} "
              f"{str(v['ok']):>16s}   {why}")
    assert can_run_isaac_sim(GPU_LEDGER[0])["ok"], "the RTX 4080 IS the documented minimum"
    t4 = can_run_isaac_sim(GPU_LEDGER[2])
    assert not t4["ok"] and len(t4["reasons"]) == 1, (
        f"the T4 produced {len(t4['reasons'])} reasons: {t4['reasons']}. It has RT Cores and "
        "16 GiB, so it must fail exactly one check — the 'not named' one. If you gave it a "
        "VRAM reason, your comparison is > rather than >=."
    )
    a100 = can_run_isaac_sim(GPU_LEDGER[4])
    assert len(a100["reasons"]) == 2, (
        f"the A100 produced {len(a100['reasons'])} reasons: {a100['reasons']}. It has 40 GiB "
        "but no RT Cores and is not named, so it fails exactly two."
    )
    print("\nexercise 5 looks right. Note the T4 row: it fails for being unnamed, NOT for"
          "\nlacking RT Cores. Check the claim, not the folklore.")


if _IS_MAIN:
    _try("exercise 5", _check_can_run_isaac_sim)

## 10. The hardware ladder, with prices that carry their receipts

If you want to go past what a CPU can teach, these are the real rungs. Every figure was
read from the seller's own page on the date shown and is recorded in `claims.yaml`.

One rung is deliberately empty. No retail price for an RTX 4080-class desktop GPU appears
here, because NVIDIA's own page rendered its price as a placeholder and three retailers
refused an unauthenticated request. Gate 12 forbids a number without a primary source, and
that rule binds the person who wrote this lesson exactly as it binds you.

In [ ]:
HARDWARE_LADDER = [
    {"rung": "a 6-DoF arm, servo kit", "cost_usd": 249.90, "kind": "buy",
     "what_it_teaches": "real actuators, real backlash, real latency",
     "source": "https://www.seeedstudio.com/SO-ARM101-Low-Cost-AI-Arm-Kit-p-6426.html"},
    {"rung": "Reachy Mini Lite", "cost_usd": 399.00, "kind": "buy",
     "what_it_teaches": "an open-source robot with a real sensing loop",
     "source": "https://huggingface.co/blog/reachy-mini"},
    {"rung": "Unitree Go2 quadruped", "cost_usd": 2800.00, "kind": "buy",
     "what_it_teaches": "contact, slip and balance on hardware",
     "source": "https://shop.unitree.com/products/unitree-go2"},
    {"rung": "Berkeley Humanoid Lite (build it)", "cost_usd": 5000.00, "kind": "build",
     "what_it_teaches": "a whole open humanoid, printed on a desktop printer",
     "source": "https://arxiv.org/abs/2504.17249"},
    {"rung": "Unitree G1 humanoid", "cost_usd": 13500.00, "kind": "buy",
     "what_it_teaches": "the real thing, and the real repair bill",
     "source": "https://shop.unitree.com/products/unitree-g1"},
    {"rung": "rent an RTX 4090, per hour", "cost_usd": 0.74, "kind": "rent",
     "what_it_teaches": "Isaac Sim, without owning the card",
     "source": "https://www.runpod.io/pricing"},
    {"rung": "rent an RTX 5090, per hour", "cost_usd": 0.99, "kind": "rent",
     "what_it_teaches": "the same, faster",
     "source": "https://www.runpod.io/pricing"},
]


def ladder_table(ladder: list = HARDWARE_LADDER) -> None:
    """Print the ladder, and compute how much rental time each purchase price would buy."""
    hourly = min(r["cost_usd"] for r in ladder if r["kind"] == "rent")
    print(f"\n{'rung':<36s} {'USD':>9s}  {'kind':<6s} what it teaches")
    for r in sorted(ladder, key=lambda row: row["cost_usd"]):
        print(f"{r['rung']:<36s} {r['cost_usd']:>9,.2f}  {r['kind']:<6s} "
              f"{r['what_it_teaches']}")
    print(f"\nCheapest rented RTX hour on the ladder: ${hourly:.2f}.")
    for r in sorted(ladder, key=lambda row: row["cost_usd"]):
        if r["kind"] != "rent":
            print(f"  the {r['rung']} budget instead buys {r['cost_usd'] / hourly:,.0f} "
                  f"GPU-hours of Isaac Sim")
    print("\nEvery source URL is in claims.yaml with the date it was read. Prices move; the"
          "\nmethod of checking them before you repeat them does not.")


if _IS_MAIN:
    ladder_table()

## 11. Common mistakes

- **Believing the T4 has no RT Cores.** It has them, and 16 GB. The reason a free tier
  cannot run Isaac Sim is that the card is not one NVIDIA names, and that the free tiers do
  not promise you any particular card. Repeating the folklore is how a wrong fact survives.
- **`model.body_mass[i] *= scale`.** It compounds. The second call on the same model gives
  you `scale²`, and the study quietly measures a machine that never existed. Set from a
  stored baseline.
- **Forgetting `mj_setConst` after changing mass.** The quantities MuJoCo derived from the
  old mass keep the old value, silently. On this model the rollout is unchanged — say that
  precisely rather than claiming a physics error you have not measured — but the derived
  bookkeeping is stale, and on a model with tendons or a solver that reads those fields it
  stops being cosmetic. The cell below shows it happening.
- **Adding sensor noise to the state instead of the observation.** Corrupting `data.qpos`
  simulates a robot that is genuinely somewhere else, not one that is *mistaken* about
  where it is. Noise belongs in what the controller reads, never in the world.
- **A latency pipeline that does not start full.** If the pipeline starts empty, the first
  steps are undelayed and you measure a smaller gap than you built.
- **Reading back `data.ctrl` to see what the motor did.** It holds what you asked for.
  `data.actuator_force` holds what you got.
- **Reporting the score on the conditions you tuned on.** It is 100% by construction. Only
  the held-out column carries information.
- **Treating survival as the only metric.** Dry friction never topples this machine and
  still multiplies the wobble several times over and pins the actuator at its limit. The
  exact multiple is printed by the sweep in section 6, computed from your own rollouts —
  this bullet does not quote it, because a number in prose is a number that can go stale.
  The gap that eventually kills you usually shows up first as a saturation number nobody
  was watching.

In [ ]:
# Watch one of those mistakes happen. Change the mass, skip mj_setConst, and read back the
# total mass MuJoCo thinks it has. Nothing here is typed; every number is measured.
_stale_model, _stale_data = load_balancer()
print(f"total mass on file          : {_stale_model.body_subtreemass[0]:.1f} kg")
_stale_model.body_mass[TORSO_ID] = BASE_TORSO_MASS * 2.0      # the edit, without mj_setConst
print(f"after doubling the torso    : {_stale_model.body_subtreemass[0]:.1f} kg   <- stale, "
      f"though body_mass now reads {_stale_model.body_mass[TORSO_ID]:.1f} kg")
mujoco.mj_setConst(_stale_model, _stale_data)
print(f"after mujoco.mj_setConst    : {_stale_model.body_subtreemass[0]:.1f} kg   <- rebuilt")

## 12. Self-check

1. Your controller survives every condition you tuned it on and falls on two thirds of a
   held-out set. What have you learned?
   - (a) the held-out set is unfairly hard
   - (b) the tuning worked, and the gap is a separate problem to solve later
   - (c) you measured the training score, which is 100% by construction, and the only
         informative number is the held-out one
   - (d) the controller needs higher gains

2. This lesson randomises torso mass and joint dry friction rather than textures and
   lighting. Which line of work is that, and why does it matter here?
   - (a) Tobin et al. (2017); appearance is what transfers
   - (b) Peng et al. (2017) dynamics randomisation; this controller reads joint angles, not
         pixels, so only the dynamics can possibly matter to it
   - (c) neither; domain randomisation only applies to vision
   - (d) both are the same technique under different names

3. A colleague says free Colab cannot run Isaac Sim because the T4 has no ray-tracing
   cores. What is wrong with that?
   - (a) nothing, it is correct
   - (b) the T4 does have RT Cores and does meet the 16 GB floor; it fails because it is not
         a GPU NVIDIA names in its requirements table, and because the free tier does not
         guarantee you any particular GPU
   - (c) Colab does not offer GPUs at all
   - (d) Isaac Sim does not need a GPU

4. You add 8 ms of control latency. Survival is unchanged, but the saturation fraction goes
   from a third of the episode to all of it. What should you conclude?
   - (a) nothing; it survived, so the latency is harmless
   - (b) the measurement is broken, because latency cannot change saturation
   - (c) the controller is now spending the entire episode against its torque limit, so it
         has no margin left for any other disturbance — the gap has already done damage that
         survival alone cannot see
   - (d) the torque limit should be raised until saturation disappears

Answers, with reasoning, are in this lesson's worked solution in the course repository.

In [ ]:
# Put your four letters here and run the cell. It marks them without revealing the answer:
# a wrong letter sends you back to the section that measured it, which is the point.
SELF_CHECK = {1: "?", 2: "?", 3: "?", 4: "?"}

_ANSWER_DIGESTS = {1: "6f62e7f0082bcf7c", 2: "c5c3917d88602bae",
                   3: "03e5e02cc8ff791f", 4: "df9689b878adb253"}
_ANSWER_SECTIONS = {
    1: "section 8 — what a held-out score is for, and what a training score is not",
    2: "section 7 — which of the two randomisation papers this lesson actually follows",
    3: "section 9 — the T4 row of the ledger you computed, not the folklore about it",
    4: "section 6 — the saturation column of the latency sweep you ran",
}


def _check_self_check(answers: dict = None) -> None:
    """Mark the four multiple-choice answers in SELF_CHECK, naming where to look again."""
    answers = SELF_CHECK if answers is None else answers
    wrong = []
    for q, digest in sorted(_ANSWER_DIGESTS.items()):
        got = str(answers.get(q, "?")).strip().lower()
        if hashlib.sha256(f"F15-L08-q{q}-{got}".encode()).hexdigest()[:16] != digest:
            wrong.append(q)
    for q in sorted(_ANSWER_DIGESTS):
        note = f"  -> re-read {_ANSWER_SECTIONS[q]}" if q in wrong else ""
        print(f"  q{q}: {'wrong' if q in wrong else 'right'}{note}")
    assert not wrong, (
        f"questions {wrong} are still wrong. Each one names the section that answers it "
        "above — go back to the measurement you ran there rather than guessing a letter."
    )
    print("self-check: all four right")


def _check_self_check_answered() -> None:
    """The self-check, through the guard: four question marks are not started, not wrong."""
    if all(str(v).strip() == "?" for v in SELF_CHECK.values()):
        raise NotImplementedError("put your four letters in SELF_CHECK above")
    _check_self_check()


if _IS_MAIN:
    _try("self-check", _check_self_check_answered)

## What you built, and where it goes next

A bench that turns each of the four reality gaps into a number, a domain-randomisation
wrapper whose benefit you measured on held-out parameters rather than assumed, and a ledger
that decides from published requirements what this course cannot give you.

The capstone (`CAPSTONE.md` in the flagship root) asks for a walk of a measured distance
without falling — and grades it under randomised parameters, not the nominal model, using
exactly the `sample_conditions` / `evaluate` contract you implemented here.

In [ ]:
# Your progress board. It reads the verdict each check cell recorded the last time it ran, so
# after you fix an exercise, re-run that exercise's check cell and then this one.
def _progress_board() -> list:
    """Print one line per exercise, then the tally. Returns the labels that came back wrong."""
    marks = {"passed": "✅", "failed": "❌", "not started": "⏳"}
    print("\nprogress board")
    for label, what in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {marks[state]} {label}  {what:<20s} {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    return [label for label, state in _STATUS.items() if state == "failed"]


if _IS_MAIN:
    _failed = _progress_board()
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends the run non-zero, so a green exit code cannot paper
    # over it. Inside a notebook kernel the same verdict is a printed line, not a traceback.
    if _failed and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(_failed))
    if _failed:
        print("checks failed: " + ", ".join(_failed) + " — each printed its reason above")